# Backtest Results — Combined V2 (Regime-Switching, One Row per Market)
**Script:** `model/backtest_all_models.py --combined-v2`  
**Source:** Actual run log (`log_backtest.txt`)  
**Architecture:** ONE row per market — uses regime detection to switch between Uptrend & Downtrend models  
**Strategies tried:** Discrete modes (long_bias/long_only/standard/bh_anchor) + Weighted (confidence/asymmetric/pyramid) + EMA smoothing + Regime gating + Volatility targeting + Trailing stops  
**Scoring:** Alpha + 0.5×Sortino + 2.0×Calmar − DD penalty

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 130
print('Libraries loaded')

## 1. Load Backtest Results from Actual Run Log

In [ ]:
# From log_backtest.txt — Combined V2 FINAL RESULTS (one row per market)
raw = [
    {'Market':'US',  'Regime':'HMM',                'UpModel':'RandomForest', 'DownModel':'LSTM',         'BestStrategy':'Mode=standard Th=0.50 Trail=0.00',                                'Return':38.10, 'BnH':33.11, 'MaxDD':-18.90, 'Sharpe':1.10, 'Sortino':1.44, 'Calmar':1.00, 'Final':13809.68},
    {'Market':'UK',  'Regime':'HMM',                'UpModel':'LSTM',         'DownModel':'SVM',          'BestStrategy':'Mode=long_bias Th=0.74 Trail=0.00',                               'Return':27.33, 'BnH':27.21, 'MaxDD':-13.43, 'Sharpe':1.16, 'Sortino':1.39, 'Calmar':1.02, 'Final':12732.61},
    {'Market':'Thai','Regime':'LogisticRegression', 'UpModel':'Transformer',  'DownModel':'CNN',          'BestStrategy':'W(asymmetric,regime_gated) ex=0.30 en=0.55 f=0.0 T=0.00 VT=0.00', 'Return':19.46, 'BnH':15.13, 'MaxDD':-28.12, 'Sharpe':0.65, 'Sortino':0.92, 'Calmar':0.37, 'Final':11945.59},
    {'Market':'Gold','Regime':'SMA200',             'UpModel':'Transformer',  'DownModel':'XGBoost',      'BestStrategy':'Mode=bh_anchor Trail=0.00',                                       'Return':100.65,'BnH':100.85,'MaxDD':-17.73, 'Sharpe':1.72, 'Sortino':1.98, 'Calmar':2.55, 'Final':20065.11},
    {'Market':'BTC', 'Regime':'XGBoost',            'UpModel':'MLP',          'DownModel':'RandomForest', 'BestStrategy':'W(pyramid,raw) ex=0.45 en=0.55 f=0.2 T=0.00 VT=0.00',             'Return':59.03, 'BnH':15.72, 'MaxDD':-43.99, 'Sharpe':0.65, 'Sortino':0.95, 'Calmar':0.41, 'Final':15902.54},
]

df = pd.DataFrame(raw)
df['Alpha'] = df['Return'] - df['BnH']
df['Beat_BnH'] = df['Alpha'] > 0

# Simplified strategy label for plotting
def simplify(s):
    if s.startswith('W('):
        if 'pyramid' in s: return 'pyramid'
        if 'asymmetric' in s: return 'asymmetric'
        if 'confidence' in s: return 'confidence'
        return 'weighted'
    s = s.split(' Trail')[0]
    return s.replace('Mode=','')
df['StrategyShort'] = df['BestStrategy'].apply(simplify)

print(f'Total markets: {len(df)}')
print(f'Beat B&H: {df["Beat_BnH"].sum()} / {len(df)}')
print(f'Average Alpha: {df["Alpha"].mean():+.2f}%')
df[['Market','Regime','UpModel','DownModel','StrategyShort','Return','BnH','Alpha','MaxDD']]

## 2. Strategy Return vs Buy & Hold

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(df))
w = 0.38

b1 = ax.bar(x - w/2, df['Return'], w, label='Strategy Return', color='#2E86AB', edgecolor='white')
b2 = ax.bar(x + w/2, df['BnH'],    w, label='Buy & Hold',     color='#E07A5F', edgecolor='white')

for bar, val in zip(b1, df['Return']):
    ax.text(bar.get_x() + bar.get_width()/2, val + 1.5,
            f'{val:+.1f}%', ha='center', fontsize=10, fontweight='bold', color='#1B4965')
for bar, val in zip(b2, df['BnH']):
    ax.text(bar.get_x() + bar.get_width()/2, val + 1.5,
            f'{val:+.1f}%', ha='center', fontsize=9.5, color='#7A2E1F')

ax.set_xticks(x)
ax.set_xticklabels(df['Market'], fontsize=11, fontweight='bold')
ax.axhline(0, color='black', linewidth=0.7)
ax.set_ylabel('Return (%)', fontsize=12)
ax.set_title('Combined V2 — Strategy Return vs Buy & Hold (Per Market)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='upper left')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('return_vs_bnh.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Alpha — Excess Return Over Buy & Hold

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
colors = ['#2E7D32' if a > 0 else '#C62828' for a in df['Alpha']]
bars = ax.bar(df['Market'], df['Alpha'], color=colors, edgecolor='white', linewidth=1.2, width=0.5)

for bar, a in zip(bars, df['Alpha']):
    offset = 1.0 if a >= 0 else -1.5
    ax.text(bar.get_x() + bar.get_width()/2, a + offset,
            f'{a:+.2f}%', ha='center', fontsize=11, fontweight='bold')

ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Alpha = Strategy − B&H (%)', fontsize=12)
ax.set_xlabel('Market', fontsize=12)
ax.set_title('Combined V2 Alpha — Excess Return vs Buy & Hold\n(Green = Beat B&H, Red = Underperform)', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('alpha_vs_bnh.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nBeat B&H: {df["Beat_BnH"].sum()} / {len(df)} ({df["Beat_BnH"].mean()*100:.0f}%)')
print(f'Average Alpha: {df["Alpha"].mean():+.2f}%')
print(f'Best Alpha: {df["Alpha"].max():+.2f}% ({df.loc[df["Alpha"].idxmax(), "Market"]})')
print(f'Worst Alpha: {df["Alpha"].min():+.2f}% ({df.loc[df["Alpha"].idxmin(), "Market"]})')

## 4. Risk-Adjusted Performance Metrics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Sharpe
ax = axes[0]
colors1 = ['#2E7D32' if s > 1 else ('#FFA000' if s > 0.5 else '#C62828') for s in df['Sharpe']]
bars = ax.bar(df['Market'], df['Sharpe'], color=colors1, edgecolor='white')
for bar, v in zip(bars, df['Sharpe']):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.03, f'{v:.2f}', ha='center', fontweight='bold')
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.6, label='Excellent (>1)')
ax.set_title('Sharpe Ratio', fontsize=12, fontweight='bold')
ax.set_ylabel('Sharpe')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Sortino
ax = axes[1]
colors2 = ['#2E7D32' if s > 1 else ('#FFA000' if s > 0.5 else '#C62828') for s in df['Sortino']]
bars = ax.bar(df['Market'], df['Sortino'], color=colors2, edgecolor='white')
for bar, v in zip(bars, df['Sortino']):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.03, f'{v:.2f}', ha='center', fontweight='bold')
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.6)
ax.set_title('Sortino Ratio (Downside-aware)', fontsize=12, fontweight='bold')
ax.set_ylabel('Sortino')
ax.grid(axis='y', alpha=0.3)

# Calmar
ax = axes[2]
colors3 = ['#2E7D32' if c > 1 else ('#FFA000' if c > 0.5 else '#C62828') for c in df['Calmar']]
bars = ax.bar(df['Market'], df['Calmar'], color=colors3, edgecolor='white')
for bar, v in zip(bars, df['Calmar']):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.05, f'{v:.2f}', ha='center', fontweight='bold')
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.6)
ax.set_title('Calmar Ratio (Return / |MaxDD|)', fontsize=12, fontweight='bold')
ax.set_ylabel('Calmar')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('risk_adjusted.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Risk vs Return Scatter

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
market_colors = {'BTC':'#F2A900','Gold':'#FFD700','Thai':'#0088CC','UK':'#012169','US':'#B22234'}

for _, r in df.iterrows():
    ax.scatter(abs(r['MaxDD']), r['Return'],
               s=400, c=market_colors[r['Market']],
               edgecolors='black', linewidths=2, alpha=0.85, zorder=3)
    ax.annotate(f"{r['Market']}\n{r['StrategyShort']}\nUp:{r['UpModel']}/Down:{r['DownModel']}",
                (abs(r['MaxDD']), r['Return']),
                xytext=(12, 0), textcoords='offset points',
                fontsize=9, va='center')
    # Add B&H reference
    ax.scatter(abs(r['MaxDD']), r['BnH'],
               s=80, c=market_colors[r['Market']],
               marker='x', alpha=0.5, zorder=2)

ax.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.6)
ax.set_xlabel('|Max Drawdown| (%)  — Risk', fontsize=12)
ax.set_ylabel('Strategy Return (%)  — Reward', fontsize=12)
ax.set_title('Risk vs Return — Combined V2 Strategy\n(circles = Strategy, X = Buy & Hold reference)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)

patches = [mpatches.Patch(color=c, label=m) for m, c in market_colors.items()]
ax.legend(handles=patches, loc='upper left', title='Market')
plt.tight_layout()
plt.savefig('risk_return_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Final Capital from $10,000 Initial

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
df_sorted = df.sort_values('Final', ascending=True).reset_index(drop=True)
colors = ['#388E3C' if v >= 10000 else '#C62828' for v in df_sorted['Final']]
bars = ax.barh(df_sorted['Market'], df_sorted['Final'],
               color=colors, edgecolor='white', linewidth=1.2)

for bar, val, up, dn in zip(bars, df_sorted['Final'], df_sorted['UpModel'], df_sorted['DownModel']):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
            f'${val:,.0f}  (Up:{up}/Down:{dn})', va='center', fontsize=10, fontweight='bold')

ax.axvline(10000, color='black', linestyle='--', linewidth=1.5, label='Initial $10,000')
ax.set_xlabel('Final Capital (USD)', fontsize=12)
ax.set_title('Final Portfolio Value per Market — Initial $10,000 (Combined V2)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')
ax.grid(axis='x', alpha=0.3)
ax.set_xlim(0, df_sorted['Final'].max() * 1.30)
plt.tight_layout()
plt.savefig('final_capital.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Strategy Mode Distribution

In [ ]:
mode_count = df['StrategyShort'].value_counts().reset_index()
mode_count.columns = ['Strategy', 'Count']

mode_palette = {
    'long_bias':'#2E86AB','long_only':'#A23B72','standard':'#F18F01',
    'bh_anchor':'#C73E1D','asymmetric':'#3B6E8F','pyramid':'#7B2D26',
    'confidence':'#5D737E','long_neutral':'#888888'
}

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(mode_count['Strategy'], mode_count['Count'],
               color=[mode_palette.get(m, '#888') for m in mode_count['Strategy']],
               edgecolor='white', linewidth=1)

for bar, cnt in zip(bars, mode_count['Count']):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f'{cnt} market(s)', va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Number of Markets', fontsize=11)
ax.set_title('Strategy Mode Selection (5 Markets Total)', fontsize=13, fontweight='bold')
ax.set_xlim(0, mode_count['Count'].max() + 1.5)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('strategy_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== Strategy Selection ===')
print(mode_count.to_string(index=False))

## 8. Champion Models per Market

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(df))

all_models = sorted(set(list(df['UpModel']) + list(df['DownModel'])))
model_colors = {
    'LSTM': '#4C72B0', 'CNN': '#DD8452', 'MLP': '#55A868',
    'Transformer': '#C44E52', 'RandomForest': '#8172B2',
    'SVM': '#937860', 'XGBoost': '#DA8BC3'
}

w = 0.35
for i, r in df.iterrows():
    ax.bar(i - w/2, 1, w, color=model_colors.get(r['UpModel'], '#888'),
           edgecolor='white', linewidth=1.5)
    ax.bar(i + w/2, 1, w, color=model_colors.get(r['DownModel'], '#888'),
           edgecolor='white', linewidth=1.5)
    ax.text(i - w/2, 0.5, r['UpModel'], ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    ax.text(i + w/2, 0.5, r['DownModel'], ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    ax.text(i - w/2, 1.05, 'Up Trend', ha='center', fontsize=8)
    ax.text(i + w/2, 1.05, 'Down Trend', ha='center', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(df['Market'], fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.2)
ax.set_yticks([])
ax.set_title('Champion Models per Market — Uptrend vs Downtrend', fontsize=13, fontweight='bold')

patches = [mpatches.Patch(color=model_colors.get(m, '#888'), label=m) for m in all_models]
ax.legend(handles=patches, loc='upper right', fontsize=9, ncol=4, title='Model')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
plt.tight_layout()
plt.savefig('champion_models.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Summary

In [ ]:
print('=' * 80)
print('Combined V2 Backtest Summary — Regime-Switching Strategy')
print('=' * 80)

summary = df[['Market','Regime','UpModel','DownModel','StrategyShort','Return','BnH','Alpha','MaxDD','Sharpe','Sortino','Calmar','Final']].copy()
summary.columns = ['Market','Regime','UpModel','DownModel','Strategy','Ret(%)','B&H(%)','Alpha(%)','DD(%)','Sharpe','Sortino','Calmar','Final($)']
print('\n--- Final Results Table ---')
print(summary.to_string(index=False))

print(f'\n--- Aggregate Metrics ---')
print(f'  Average Return:           {df["Return"].mean():+.2f}%')
print(f'  Average Buy & Hold:       {df["BnH"].mean():+.2f}%')
print(f'  Average Alpha:            {df["Alpha"].mean():+.2f}%')
print(f'  Average MaxDD:            {df["MaxDD"].mean():.2f}%')
print(f'  Average Sharpe:           {df["Sharpe"].mean():.2f}')
print(f'  Average Sortino:          {df["Sortino"].mean():.2f}')
print(f'  Average Calmar:           {df["Calmar"].mean():.2f}')
print(f'  Cases Beating B&H:        {df["Beat_BnH"].sum()} / {len(df)} ({df["Beat_BnH"].mean()*100:.0f}%)')
print(f'  Best Performance:         {df.loc[df["Return"].idxmax(),"Market"]} = {df["Return"].max():+.2f}%')
print(f'  Highest Alpha:            {df.loc[df["Alpha"].idxmax(),"Market"]} = {df["Alpha"].max():+.2f}%')
print(f'  Lowest Drawdown:          {df.loc[df["MaxDD"].idxmax(),"Market"]} = {df["MaxDD"].max():.2f}%')
print(f'  Worst Drawdown:           {df.loc[df["MaxDD"].idxmin(),"Market"]} = {df["MaxDD"].min():.2f}%')
print(f'  Total Final Capital:      ${df["Final"].sum():,.2f}')
print(f'  Total Initial Capital:    ${len(df) * 10000:,.2f}')
print(f'  Aggregate Return:         {(df["Final"].sum() / (len(df) * 10000) - 1) * 100:+.2f}%')

print(f'\n--- Champion Strategy Distribution ---')
print(df['StrategyShort'].value_counts().to_string())
print('=' * 80)